# Reshaping and Combining Records

*Distinguish concatenation, aggregation, pivoting, and joining using the same score records.*


## Combine Compatible Batches into Long Data

In long data, each row represents one student–subject observation. The two
original score batches share a schema. `pd.concat(..., ignore_index=True)`
stacks their rows without calculating a summary.


In [1]:
import csv
import pandas as pd

first_lines = [
    "Jiyoung,100,Mathematics",
    "Hojun,90,English",
    "Hana,85,English",
    "Kihoon,95,Mathematics",
]
second_lines = [
    "Jiyoung,78,English",
    "Hojun,95,Mathematics",
    "Hana,77,Mathematics",
    "Kihoon,86,English",
]


def parse_scores(lines):
    return pd.DataFrame(
        [
            {"name": name, "subject": subject, "score": int(score)}
            for name, score, subject in csv.reader(lines)
        ]
    )


scores_df = pd.concat(
    [parse_scores(first_lines), parse_scores(second_lines)], ignore_index=True
)
print("Long score records:")
print(scores_df.to_string(index=False))
print(
    "Student-subject keys are unique:",
    not scores_df.duplicated(["name", "subject"]).any(),
)


Long score records:
   name     subject  score
Jiyoung Mathematics    100
  Hojun     English     90
   Hana     English     85
 Kihoon Mathematics     95
Jiyoung     English     78
  Hojun Mathematics     95
   Hana Mathematics     77
 Kihoon     English     86
Student-subject keys are unique: True


The combined table has eight observations, two for each of four students.
Every student–subject pair occurs once, which permits an unambiguous pivot.


## Aggregation Reduces Observations to Summaries

`.groupby()` forms groups. Named aggregation gives each result column a clear
meaning. A count and a mean describe a subject, while individual scores are
no longer present in that result.


In [2]:
subject_report = scores_df.groupby("subject", as_index=False).agg(
    score_count=("score", "count"), mean_score=("score", "mean")
)
print("Subject summaries:")
print(subject_report.to_string(index=False))


Subject summaries:
    subject  score_count  mean_score
    English            4       84.75
Mathematics            4       91.75


Both subjects have four scores. English has mean 84.75 and Mathematics
has mean 91.75. Two summary rows replace eight observations in this view.


## Pivoting Changes the Arrangement

`.pivot(index="name", columns="subject", values="score")` places subjects in
columns without averaging their values. A duplicate student–subject key would
need a resolution rule before this pivot. `.melt()` converts the wide view back
to long form. [pandas reshaping reference](https://pandas.pydata.org/docs/user_guide/reshaping.html).


In [3]:
wide_df = scores_df.pivot(index="name", columns="subject", values="score")
ordered_wide_df = wide_df.sort_values(["Mathematics", "name"], ascending=[False, True])
restored_long_df = wide_df.reset_index().melt(
    id_vars="name", var_name="subject", value_name="score"
)
keys = ["name", "subject"]
same_records = (
    restored_long_df.sort_values(keys)
    .reset_index(drop=True)
    .equals(
        scores_df[["name", "subject", "score"]].sort_values(keys).reset_index(drop=True)
    )
)
print("Wide scores ordered by Mathematics:")
print(ordered_wide_df)
print("Long records preserved:", same_records)


Wide scores ordered by Mathematics:
subject  English  Mathematics
name                         
Jiyoung       78          100
Hojun         90           95
Kihoon        86           95
Hana          85           77
Long records preserved: True


Jiyoung leads Mathematics with 100. Hojun and Kihoon both have 95, so their
names resolve the tie. The `True` comparison confirms that pivoting and melting
preserve all eight scores. Sorting the pivot is a separate operation.


## Joining Adds Related Fields

A lookup supplies a city for each student. A left merge keeps every score;
`validate="many_to_one"` requires each lookup name to occur at most once.
Matching by key makes the lookup row order irrelevant.


In [4]:
roster_df = pd.DataFrame(
    {
        "name": ["Hana", "Hojun", "Jiyoung", "Kihoon"],
        "city": ["Daegu", "Busan", "Seoul", "Seoul"],
    }
)
enriched_df = scores_df.merge(roster_df, on="name", how="left", validate="many_to_one")
print("Scores with city:")
print(enriched_df.to_string(index=False))
print("Rows before and after:", len(scores_df), len(enriched_df))


Scores with city:
   name     subject  score  city
Jiyoung Mathematics    100 Seoul
  Hojun     English     90 Busan
   Hana     English     85 Daegu
 Kihoon Mathematics     95 Seoul
Jiyoung     English     78 Seoul
  Hojun Mathematics     95 Busan
   Hana Mathematics     77 Daegu
 Kihoon     English     86 Seoul
Rows before and after: 8 8


The join attaches a city to each of eight score rows. It does not add new
scores or collapse existing observations.


Concatenate to add compatible records, aggregate to summarize groups, pivot to rearrange values, and join to attach related fields. Track what one row means after each operation.
